# Analiza klientów

Celem tego notebooka jest analiza klientów sklepu internetowego.

Analiza obejmuje:

- identyfikację klientów generujących najwyższy przychód
- analizę liczby zamówień na klienta
- analizę średniej wartości zamówienia
- prostą segmentację klientów

Wyniki tej analizy pomagają lepiej zrozumieć strukturę bazy klientów oraz wskazać klientów o najwyższej wartości biznesowej.

## Import bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

## Wczytanie danych

In [ ]:
df = pd.read_csv("../data/processed/sales_cleaned.csv")

df.head()

## Przygotowanie danych na poziomie klienta

Aby przeprowadzić analizę klientów, agregujemy dane do poziomu klienta.

Dla każdego klienta obliczamy:

- liczbę zamówień
- całkowity przychód
- średnią wartość zamówienia

In [ ]:
customer_summary = (
    df.groupby("customer_unique_id")
    .agg(
        orders_count=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
        avg_order_value=("order_value", "mean")
    )
    .sort_values("total_revenue", ascending=False)
    .reset_index()
)

customer_summary.head()


## Klienci generujący najwyższy przychód

In [ ]:
top_customers_revenue = customer_summary.head(10)

top_customers_revenue

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_customers_revenue,
    x="total_revenue",
    y="customer_unique_id"
)

plt.title("Top 10 klientów według przychodu")
plt.tight_layout()

plt.savefig("../reports/figures/top_customers_revenue.png")
plt.show()

### Interpretacja

Niewielka grupa klientów może odpowiadać za ponadprzeciętną część przychodu.

Tacy klienci są szczególnie istotni z perspektywy działań lojalnościowych i utrzymania klienta.

## Klienci z największą liczbą zamówień

In [ ]:
top_customers_orders = (
    customer_summary
    .sort_values("orders_count", ascending=False)
    .head(10)
)

top_customers_orders

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_customers_orders,
    x="orders_count",
    y="customer_unique_id"
)

plt.title("Top 10 klientów według liczby zamówień")
plt.tight_layout()

plt.savefig("../reports/figures/top_customers_orders.png")
plt.show()

### Interpretacja

Klienci z największą liczbą zamówień nie zawsze są tymi samymi klientami, którzy generują najwyższy przychód.

Porównanie tych dwóch rankingów pozwala odróżnić klientów często kupujących od klientów składających droższe zamówienia.

## Rozkład średniej wartości zamówienia

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    customer_summary["avg_order_value"],
    bins=50
)

plt.title("Rozkład średniej wartości zamówienia na klienta")
plt.tight_layout()

plt.savefig("../reports/figures/customer_avg_order_value_distribution.png")
plt.show()

### Interpretacja

Rozkład średniej wartości zamówienia pokazuje, czy większość klientów składa zamówienia o podobnej wartości, czy też występuje duże zróżnicowanie.

## Segmentacja klientów

Tworzymy prostą segmentację klientów na podstawie całkowitego przychodu wygenerowanego przez klienta.

In [ ]:
customer_summary["segment"] = pd.cut(
    customer_summary["total_revenue"],
    bins=[0, 100, 500, 1000, customer_summary["total_revenue"].max()],
    labels=["low", "medium", "high", "vip"],
    include_lowest=True
)

customer_summary["segment"].value_counts()

In [ ]:
segment_counts = (
    customer_summary["segment"]
    .value_counts()
    .reset_index()
)

segment_counts.columns = ["segment", "customers_count"]

segment_counts

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=segment_counts,
    x="segment",
    y="customers_count"
)

plt.title("Liczba klientów w segmentach")
plt.tight_layout()

plt.savefig("../reports/figures/customer_segments.png")
plt.show()

### Interpretacja

Segmentacja pokazuje strukturę klientów według ich wartości biznesowej.

Może być wykorzystana do:

- personalizacji ofert
- planowania kampanii marketingowych
- działań lojalnościowych

## Rozkład liczby zamówień na klienta

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    customer_summary["orders_count"],
    bins=30
)

plt.title("Rozkład liczby zamówień na klienta")
plt.tight_layout()

plt.savefig("../reports/figures/customer_orders_distribution.png")
plt.show()

### Interpretacja

Rozkład liczby zamówień pokazuje, czy większość klientów kupuje jednorazowo, czy sklep ma również grupę klientów powracających.

## Podsumowanie

Analiza klientów pozwoliła zidentyfikować:

- klientów generujących najwyższy przychód
- klientów składających największą liczbę zamówień
- strukturę średniej wartości zamówień
- segmenty klientów według wartości biznesowej

W kolejnym etapie projektu przeprowadzona zostanie analiza koszyka zakupowego.